Final test for Unet with K Fold

In [2]:
import os
import glob
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torchvision import transforms
from PIL import Image
from sklearn.model_selection import KFold
import matplotlib.pyplot as plt

# ------------------------
# 1) تعریف برخی معیارهای کمکی
# ------------------------

def dice_coefficient(pred, target, epsilon=1e-6):
    """
    Dice = 2 * (pred ∩ target) / (pred + target)
    خروجی مدل logits است، لذا ابتدا آن را سیگموید می‌کنیم و آنگاه باینری‌سازی
    """
    pred = torch.sigmoid(pred)
    pred = (pred > 0.5).float()
    intersection = (pred * target).sum(dim=(1, 2, 3))
    union = pred.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3)) + epsilon
    dice = (2.0 * intersection) / union
    return dice.mean()

def iou_coefficient(pred, target, epsilon=1e-6):
    """
    IoU = (pred ∩ target) / (pred ∪ target)
    """
    pred = torch.sigmoid(pred)
    pred = (pred > 0.5).float()
    intersection = (pred * target).sum(dim=(1,2,3))
    union = (pred + target - pred*target).sum(dim=(1,2,3)) + epsilon
    iou = intersection / union
    return iou.mean()

def precision_score(pred, target, epsilon=1e-6):
    """
    Precision = TP / (TP + FP)
    """
    pred = torch.sigmoid(pred)
    pred = (pred > 0.5).float()
    tp = (pred * target).sum(dim=(1,2,3))
    fp = (pred * (1 - target)).sum(dim=(1,2,3)) + epsilon
    precision = tp / (tp + fp)
    return precision.mean()

def recall_score(pred, target, epsilon=1e-6):
    """
    Recall = TP / (TP + FN)
    """
    pred = torch.sigmoid(pred)
    pred = (pred > 0.5).float()
    tp = (pred * target).sum(dim=(1,2,3))
    fn = ((1 - pred) * target).sum(dim=(1,2,3)) + epsilon
    recall = tp / (tp + fn)
    return recall.mean()

# ------------------------
# 2) تنظیم seed
# ------------------------
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ------------------------
# 3) تعریف دیتاست + آگومنت هماهنگ
# ------------------------
class CorneaDataset(Dataset):
    def __init__(self, images_list, masks_list, transform=None):
        self.images_list = images_list
        self.masks_list = masks_list
        self.transform = transform

    def __len__(self):
        return len(self.images_list)

    def __getitem__(self, idx):
        img_path = self.images_list[idx]
        mask_path = self.masks_list[idx]

        image = Image.open(img_path).convert("RGB")  # اگر تصاویر رنگی‌اند
        mask = Image.open(mask_path).convert("L")    # ماسک معمولاً خاکستری یا باینری

        if self.transform:
            image, mask = self.transform(image, mask)

        # باینری کردن ماسک (اگر داده‌های شما 0 و 255 هستند)
        mask = (mask > 0.5).float()

        return image, mask

class JointTransform:
    """
    کلاسی برای اعمال یکسری ترنسفورم ثابت و تصادفی به صورت همگام روی (image, mask).
    """
    def __init__(self, resize=(256, 256), h_flip=0.5, v_flip=0.5, rot=15, is_train=True):
        # برای داده‌های تست، فقط resize و ToTensor
        self.is_train = is_train
        self.resize = transforms.Resize(resize)
        self.to_tensor = transforms.ToTensor()
        
        # اگر بخواهیم data augmentation تصادفی هم داشته باشیم
        self.h_flip_prob = h_flip
        self.v_flip_prob = v_flip
        self.rot_degree = rot

    def __call__(self, img, mask):
        # ابتدا ریسایز می‌کنیم
        img = self.resize(img)
        mask = self.resize(mask)

        if self.is_train:
            # اعمال flip افقی به صورت تصادفی
            if random.random() < self.h_flip_prob:
                img = transforms.functional.hflip(img)
                mask = transforms.functional.hflip(mask)
            # اعمال flip عمودی به صورت تصادفی
            if random.random() < self.v_flip_prob:
                img = transforms.functional.vflip(img)
                mask = transforms.functional.vflip(mask)
            # اعمال rotation تصادفی
            angle = random.uniform(-self.rot_degree, self.rot_degree)
            img = transforms.functional.rotate(img, angle)
            mask = transforms.functional.rotate(mask, angle)

        # در نهایت تبدیل به تنسور
        img = self.to_tensor(img)
        mask = self.to_tensor(mask)
        return img, mask

# ------------------------
# 4) آدرس فولدر تصاویر
# ------------------------
images_dir = r"C:\Users\Meta Pc\PycharmProjects\pythonProject\myproject\payanName\my_work\segment\images"
labels_dir = r"C:\Users\Meta Pc\PycharmProjects\pythonProject\myproject\payanName\segment\corneaLabels"

images_list = sorted(glob.glob(os.path.join(images_dir, "*.*")))
masks_list  = sorted(glob.glob(os.path.join(labels_dir, "*.*")))
assert len(images_list) == len(masks_list), "تعداد تصاویر با ماسک‌ها برابر نیست."

# کل داده را به‌صورت یک دیتاست کلی درنظر می‌گیریم.
# در مرحله kFold آنها را تقسیم می‌کنیم.
full_dataset = CorneaDataset(
    images_list,
    masks_list,
    transform=None  # فعلاً اینجا چیزی نمی‌دهیم، در هر Fold جداگانه اعمال می‌کنیم
)

# ------------------------
# 5) تعریف مدل ساده U-Net
# ------------------------
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(DoubleConv, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.conv(x)

class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1):
        super(UNet, self).__init__()

        self.conv_down1 = DoubleConv(in_channels, 64)
        self.conv_down2 = DoubleConv(64, 128)
        self.conv_down3 = DoubleConv(128, 256)
        self.conv_down4 = DoubleConv(256, 512)
        self.maxpool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.bottleneck = DoubleConv(512, 1024)
        
        self.uptrans1 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.conv_up1 = DoubleConv(1024, 512)
        self.uptrans2 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.conv_up2 = DoubleConv(512, 256)
        self.uptrans3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.conv_up3 = DoubleConv(256, 128)
        self.uptrans4 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.conv_up4 = DoubleConv(128, 64)

        self.output = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        # Encoder
        x1 = self.conv_down1(x)
        x2 = self.maxpool(x1)

        x2 = self.conv_down2(x2)
        x3 = self.maxpool(x2)

        x3 = self.conv_down3(x3)
        x4 = self.maxpool(x3)

        x4 = self.conv_down4(x4)
        x5 = self.maxpool(x4)

        # Bottleneck
        x5 = self.bottleneck(x5)

        # Decoder
        x6 = self.uptrans1(x5)
        x6 = torch.cat([x4, x6], dim=1)
        x6 = self.conv_up1(x6)

        x7 = self.uptrans2(x6)
        x7 = torch.cat([x3, x7], dim=1)
        x7 = self.conv_up2(x7)

        x8 = self.uptrans3(x7)
        x8 = torch.cat([x2, x8], dim=1)
        x8 = self.conv_up3(x8)

        x9 = self.uptrans4(x8)
        x9 = torch.cat([x1, x9], dim=1)
        x9 = self.conv_up4(x9)

        out = self.output(x9)
        return out

# ------------------------
# 6) حلقه اصلی K-Fold
# ------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
k = 5
num_epochs = 20
batch_size = 8
lr = 1e-3

# برای ذخیرهٔ میانگین معیارها در کل فولدها
all_folds_dice = []
all_folds_iou = []
all_folds_prec = []
all_folds_recall = []

kf = KFold(n_splits=k, shuffle=True, random_state=seed)

fold_index = 1
for train_index, val_index in kf.split(images_list):
    print(f"\n======= Fold {fold_index} of {k} =======")

    # ایندکس‌های مربوط به Train و Val را جدا می‌کنیم
    train_images = [images_list[i] for i in train_index]
    train_masks  = [masks_list[i] for i in train_index]

    val_images = [images_list[i] for i in val_index]
    val_masks  = [masks_list[i] for i in val_index]

    # دیتاست‌های مربوطه با آگومنت مناسب
    train_dataset = CorneaDataset(
        train_images,
        train_masks,
        transform=JointTransform(resize=(256,256), h_flip=0.5, v_flip=0.5, rot=15, is_train=True)
    )
    val_dataset = CorneaDataset(
        val_images,
        val_masks,
        transform=JointTransform(resize=(256,256), h_flip=0.0, v_flip=0.0, rot=0, is_train=False)
    )

    # دیتالودر
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
    val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False, num_workers=0)

    # تعریف مدل و optimizer و loss
    model = UNet(in_channels=3, out_channels=1).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss()

    for epoch in range(num_epochs):
        model.train()
        train_loss_epoch = 0
        train_batches = 0

        for imgs, masks in train_loader:
            imgs, masks = imgs.to(device), masks.to(device)

            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, masks)
            loss.backward()
            optimizer.step()

            train_loss_epoch += loss.item()
            train_batches += 1

        avg_train_loss = train_loss_epoch / train_batches

        # ارزیابی روی val
        model.eval()
        val_loss_epoch = 0
        val_dice_epoch = 0
        val_iou_epoch = 0
        val_prec_epoch = 0
        val_recall_epoch = 0
        val_batches = 0

        with torch.no_grad():
            for imgs, masks in val_loader:
                imgs, masks = imgs.to(device), masks.to(device)
                outputs = model(imgs)
                loss = criterion(outputs, masks)

                val_loss_epoch += loss.item()
                val_dice_epoch += dice_coefficient(outputs, masks).item()
                val_iou_epoch += iou_coefficient(outputs, masks).item()
                val_prec_epoch += precision_score(outputs, masks).item()
                val_recall_epoch += recall_score(outputs, masks).item()

                val_batches += 1

        avg_val_loss   = val_loss_epoch / val_batches
        avg_val_dice   = val_dice_epoch / val_batches
        avg_val_iou    = val_iou_epoch / val_batches
        avg_val_prec   = val_prec_epoch / val_batches
        avg_val_recall = val_recall_epoch / val_batches


        print(f" Epoch [{epoch+1}/{num_epochs}]  "
                f"TrainLoss: {avg_train_loss:.4f}  "
                f"ValLoss: {avg_val_loss:.4f}  "
                f"ValDice: {avg_val_dice:.4f}  "
                f"ValIoU: {avg_val_iou:.4f}")

    # پس از اتمام آموزش این فولد، معیارهای نهایی را ذخیره می‌کنیم
    all_folds_dice.append(avg_val_dice)
    all_folds_iou.append(avg_val_iou)
    all_folds_prec.append(avg_val_prec)
    all_folds_recall.append(avg_val_recall)

    # ذخیرهٔ مدل این فولد
    model_path = f"unet_fold{fold_index}.pth"
    torch.save(model.state_dict(), model_path)
    print(f"=> Model of Fold {fold_index} saved to: {model_path}")

    # حالا می‌توانیم روی دیتای Train خود این فولد یا Val خود این فولد دوباره ارزیابی کنیم
    # (اگر خواستید روی کل دیتای Train یا هر جای دیگری ارزیابی کنید، دیتاست و دیتالودر متناظر بسازید.)
    
    fold_index += 1

# ------------------------
# 7) محاسبهٔ میانگین معیارها در کل فولدها
# ------------------------
mean_dice   = np.mean(all_folds_dice)
mean_iou    = np.mean(all_folds_iou)
mean_prec   = np.mean(all_folds_prec)
mean_recall = np.mean(all_folds_recall)

print("\n======= Cross Validation Results (Average over 5 folds) =======")
print(f"Dice   : {mean_dice:.4f}")
print(f"IoU    : {mean_iou:.4f}")
print(f"Precision: {mean_prec:.4f}")
print(f"Recall   : {mean_recall:.4f}")

# ----------------------------------------------------------------
# (اختیاری) آموزش مدل نهایی روی کل داده‌ها و ذخیره آن
# ----------------------------------------------------------------

# اگر بخواهید در انتها مدلی روی کل داده‌ها train کنید:
final_train_dataset = CorneaDataset(
    images_list,
    masks_list,
    transform=JointTransform(resize=(256,256), h_flip=0.5, v_flip=0.5, rot=15, is_train=True)
)
final_loader = DataLoader(final_train_dataset, batch_size=batch_size, shuffle=True)

final_model = UNet().to(device)
optimizer = optim.Adam(final_model.parameters(), lr=lr)
criterion = nn.BCEWithLogitsLoss()

final_epochs = 10
for epoch in range(final_epochs):
    final_model.train()
    running_loss = 0
    c = 0
    for imgs, masks in final_loader:
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        outputs = final_model(imgs)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        c += 1
    print(f"[Final Training] Epoch [{epoch+1}/{final_epochs}] Loss: {running_loss/c:.4f}")

# ذخیره مدل نهایی
torch.save(final_model.state_dict(), "unet_final.pth")
print("=> Final model trained on ALL data is saved as unet_final.pth")

# ----------------------------------------------------------------
# 8) انتخاب 10 تصویر تصادفی از داده‌های تست (مثلاً فولد آخر) و نمایش خروجی مدل
# ----------------------------------------------------------------

# فرض کنیم بخواهیم از آخرین فولد (val_images, val_masks) برای نمایش نمونه‌ها استفاده کنیم.
# اگر دیتای تست جداگانه دارید، می‌توانید همان را بسازید.
test_dataset = CorneaDataset(
    val_images,
    val_masks,
    transform=JointTransform(resize=(256,256), h_flip=0.0, v_flip=0.0, rot=0, is_train=False)
)

# مدل نهایی را لود کنیم (اگر بخواهیم از مدل نهایی استفاده کنیم)
final_model.eval()

# 10 تا ایندکس تصادفی انتخاب می‌کنیم
test_indices = list(range(len(test_dataset)))
random.shuffle(test_indices)
sample_indices = test_indices[:10]

for i in sample_indices:
    image, mask = test_dataset[i]
    # چون دیتالودر نداریم، خودمان یک چنل batch اضافه می‌کنیم
    inp = image.unsqueeze(0).to(device)
    with torch.no_grad():
        pred = final_model(inp)
    pred_sig = torch.sigmoid(pred)
    pred_bin = (pred_sig > 0.5).float()

    # تبدیل به CPU و numpy
    image_np = image.permute(1,2,0).numpy()       # [H,W,C]
    mask_np  = mask.squeeze().numpy()            # [H,W]
    pred_np  = pred_bin.squeeze().cpu().numpy()  # [H,W]

    # نمایش هر کدام در شکل جدا (بدون ساب‌پلات، مطابق دستور)
    plt.figure()
    plt.imshow(image_np)
    plt.title("Original Image")
    plt.show()

    plt.figure()
    plt.imshow(mask_np, cmap="gray")
    plt.title("Ground Truth Mask")
    plt.show()

    plt.figure()
    plt.imshow(pred_np, cmap="gray")
    plt.title("Predicted Mask")
    plt.show()

print("All done!")
